# Module 3: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the in-store shopping assistant, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, score the agent end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
4. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, re, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()


def qualify(name: str) -> str:
    """Namespace a shared resource name per user so parallel workshop runs
    don't collide on datasets / Context Hub repos / experiments."""
    who = (os.environ.get("USER") or os.environ.get("USERNAME") or "workshop")
    slug = re.sub(r"[^a-z0-9-]+", "-", who.lower()).strip("-") or "workshop"
    return f"{name}-{slug}"


print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))

## The Agent: In-Store Shopping Assistant

The agent we observe and improve in this module is a small **Deep Agent** — the same in-store shopping assistant we built in Module 2 and deployed in the deploy module. It helps a shopper move through the store:

- **Find items** — look up an item's aisle, stock status, and price in the store directory.
- **Handle out-of-stock** — check stock and suggest a practical in-store substitution.
- **Build a route** — assemble the list into an aisle-ordered walking route so the shopper walks the store once.

It's a supervisor + an `item-finder-agent` subagent, wired to store tools — `store_directory`, `check_stock`, `price_lookup`, `find_substitution` — plus the model provider's native `web_search` for recipes and open-ended product questions.

Rather than treat it as a finished black box, we'll **compose it in three pieces** so each part is inspectable and versioned:

1. **Tools** — the store functions above (defined in `agents/research_agent.py`).
2. **Memory** — an `AGENTS.md` operating manual: the assistant's workflow and rules.
3. **Skills** — a `store-route` output format it can reach for.

We'll put the manual and skills in a **LangSmith Context Hub** repo — versioned in the Hub, pulled at build time — instead of pinning them to a local checkout. The next few cells walk through it. (We keep the agent lightweight — no HITL, no disk writes — so evaluation runs don't pause or leak files.)

### Compose it: inspect the manual and skills

The operating manual and skill files live under `agents/deep_agent/` — the same source of truth the deployable shopping assistant uses. Let's read them before we ship them to the Hub.

In [ ]:
from langsmith.schemas import FileEntry

# Read the shopping assistant's operating manual + skills from the deployable
# agent's directory — the same source of truth the deployed agent uses.
DEEP_AGENT_DIR = project_root / "agents" / "deep_agent"
agents_md = (DEEP_AGENT_DIR / "AGENTS.md").read_text()
skill_files = {
    f"skills/{p.parent.name}/SKILL.md": p.read_text()
    for p in DEEP_AGENT_DIR.glob("skills/*/SKILL.md")
}

print("AGENTS.md (first lines):")
print("\n".join(agents_md.splitlines()[:6]), "\n")
print("Skills:", ", ".join(sorted(skill_files)))

context_repo = f"-/{qualify('shopping-assistant-context')}"   # "-" = your default workspace tenant; per-user repo

# One file entry per path: the manual plus every skill's SKILL.md.
files = {"AGENTS.md": FileEntry(content=agents_md)}
files.update({path: FileEntry(content=body) for path, body in skill_files.items()})

commit_url = client.push_agent(
    context_repo,
    files=files,
    description="In-store shopping assistant manual + skills",
)
print("Context Hub commit (click to open):", commit_url)

### Compose it: build the agent from Context Hub

Now build the agent with `context_repo=` set. The factory mounts that repo at `/context/` and reads its memory from `/context/AGENTS.md` and its skills from `/context/skills/` — so the manual and skills come from the Hub, not the local disk. We build once and reuse this `agent` for the warm-up and every eval below.

In [ ]:
from agents.research_agent import build_shopping_agent

# Build once against the Context Hub repo we just pushed.
agent = build_shopping_agent(context_repo=context_repo)

# Peek at the structure: tools + subagent, plus the skills & memory middleware
# that the manual and skills add to the graph.
print("Graph nodes:", list(agent.get_graph().nodes))

quick = agent.invoke(
    {"messages": [{"role": "user", "content": "What aisle is the salsa in, and is it in stock?"}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("\nSample answer:\n" + quick["messages"][-1].text)

### The shape of the agent

Before we observe or evaluate it, let's *see* the agent: its graph and the tools it can call.
The supervisor delegates to an `item-finder-agent` subagent via the `task` tool, and the store
tools (`store_directory`, `check_stock`, `price_lookup`, `find_substitution`) run inside the tool
node, alongside the native `web_search`. The Mermaid diagram below is rendered live from the
compiled graph.

In [ ]:
from IPython.display import Image, display

# Render the compiled agent graph. draw_mermaid_png() calls a remote renderer,
# so fall back to the Mermaid text diagram if that isn't reachable.
graph = agent.get_graph()
try:
    display(Image(graph.draw_mermaid_png()))
except Exception as e:
    print(f"(PNG render unavailable: {e})\n")
    print(graph.draw_mermaid())

# Inventory the tools the agent can call, so teams see its surface area.
from agents.research_agent import (
    store_directory, check_stock, price_lookup, find_substitution,
)
print("\nTools available to the agent:")
for tool_fn in [store_directory, check_stock, price_lookup, find_substitution]:
    print(f"  - {tool_fn.name}: {tool_fn.description.splitlines()[0]}")
print("  - web_search: the model provider's native web search (recipes, product info)")

## Compare two models: efficacy vs. cost

Before diving into LangSmith's features, a practical first question for any team shipping an
agent: **which model should power it?** A larger model is usually more capable but slower and
pricier; a smaller model is cheaper and faster but may cut corners. LangSmith experiments make
this an evidence-based decision instead of a guess.

We run the **same in-store shopping agent** over the **same tasks** with two different-sized
models, then compare:

- **Efficacy** — a strict `correctness` score from an LLM-as-judge (did the agent nail *every* part?).
- **Cost & speed** — total tokens and wall-clock latency per run, pulled back from the traces.

The tasks are deliberately **multi-step** — chain several store lookups, then reason or do arithmetic
over the results (total up a basket, count what's out of stock, compare two lists). That's exactly
where a smaller model tends to skip an item or miscompute, so the strict judge separates the two
models instead of scoring both 100%.

We compare **`gpt-5.6-terra`** (larger/default) against **`gpt-5.4-mini`** (smaller/cheaper).
Each run is a LangSmith experiment, so you can open both in the UI and diff them side by side.

In [ ]:
from langchain.chat_models import init_chat_model
from agents.research_agent import build_shopping_agent

# The two models to compare: larger/default vs. smaller/cheaper.
# Both route through the LangSmith Gateway like the rest of the workshop.
MODELS_TO_COMPARE = {
    "gpt-5.6-terra": "large (default)",
    "gpt-5.4-mini": "small (cheaper)",
}

def make_model(model_name):
    return init_chat_model(
        model=model_name,
        model_provider="openai",
        base_url="https://gateway.smith.langchain.com/openai",
        use_responses_api=True,
        api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
    )

# Tasks chosen to SEPARATE the models: each needs several chained store lookups
# plus reasoning/arithmetic over the results, where a smaller model tends to
# skip an item, miscompute a total, or miss an out-of-stock item. `required`
# lists the concrete facts a complete, correct answer must contain -- the judge
# checks every one. (Catalog: ground beef and lettuce are OUT OF STOCK.)
cmp_examples = [
    {
        "inputs": {"query": (
            "For a taco run I need tortillas, ground beef, salsa, and shredded cheese. "
            "Give me each item's aisle and price, flag anything out of stock, and total "
            "up only the items I can actually buy today."
        )},
        "outputs": {"reference_answer": (
            "Must give aisle + price for ALL FOUR items (tortillas $2.49 aisle 7, ground beef "
            "$6.99 aisle M3, salsa $2.79 aisle 6, shredded cheese $3.99 aisle D3), flag ground "
            "beef as OUT OF STOCK, and total ONLY the in-stock items = $2.49 + $2.79 + $3.99 = "
            "$9.27. A correct answer names all four, flags ground beef, and gives the $9.27 total."
        )},
    },
    {
        "inputs": {"query": (
            "Check stock for ground beef, lettuce, salsa, and rice. For anything out of stock, "
            "give me the substitution. How many of the four are out of stock?"
        )},
        "outputs": {"reference_answer": (
            "Must check all four: ground beef OUT (sub: ground turkey), lettuce OUT (sub: shredded "
            "cabbage), salsa in stock, rice in stock. Count of out-of-stock = 2, with the correct "
            "two substitutions. All four verdicts, both subs, and the count of 2 must be correct."
        )},
    },
    {
        "inputs": {"query": (
            "Compare two baskets: Basket A = milk + eggs + rice; Basket B = salsa + tomatoes + "
            "black beans. Which basket is cheaper and by how much? Show both totals."
        )},
        "outputs": {"reference_answer": (
            "Must price BOTH baskets: A = milk $3.49 + eggs $2.99 + rice $2.19 = $8.67; "
            "B = salsa $2.79 + tomatoes $0.99 + black beans $1.29 = $5.07; Basket B is cheaper "
            "by $3.60. Both totals AND the difference must be correct."
        )},
    },
    {
        "inputs": {"query": (
            "I want tortillas, ground beef, and avocado. Build an aisle-ordered route, and for any "
            "out-of-stock item tell me the substitution and where to find IT."
        )},
        "outputs": {"reference_answer": (
            "Must order by aisle (avocado aisle 1 Produce first, tortillas aisle 7, ground beef "
            "aisle M3 Meat), flag ground beef OUT OF STOCK with substitution ground turkey, and "
            "note the substitution is also in the Meat section. Route order, the out-of-stock flag, "
            "and the substitution must all be correct."
        )},
    },
]

cmp_dataset_name = qualify("model-comparison-evals")
if client.has_dataset(dataset_name=cmp_dataset_name):
    client.delete_dataset(dataset_id=client.read_dataset(dataset_name=cmp_dataset_name).id)
cmp_dataset = client.create_dataset(cmp_dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in cmp_examples],
    outputs=[e["outputs"] for e in cmp_examples],
    dataset_id=cmp_dataset.id,
)
print(f"Comparison dataset '{cmp_dataset_name}' ready with {len(cmp_examples)} examples.")

### A STRICT correctness judge for the comparison.
Unlike a lenient 'good enough' grader, this one fails the response if ANY required element is missing or any number is wrong -- which is what surfaces a smaller model skipping an item or miscomputing. A small model keeps judging cheap and consistent across runs.

In [ ]:
class CmpGrade(TypedDict):
    """Strict score: does the response contain every required element, all correct?"""
    score: bool
    reasoning: str

cmp_judge_prompt = (
    "You are a STRICT grader for an in-store shopping assistant.\n"
    "The success rubric lists the specific elements a complete, correct answer must "
    "contain (aisles, prices, stock flags, substitutions, totals, counts, differences).\n\n"
    "Mark score=True ONLY IF the response includes EVERY required element AND every "
    "number/verdict is correct. Mark score=False if the response omits any required "
    "element, gets any price, total, count, or stock verdict wrong, stops early, or only "
    "partially answers a multi-part request. Do not give credit for being 'close.'\n"
    "In one sentence, state exactly which required elements are present/correct and "
    "which are missing/wrong."
)
cmp_judge = make_model("gpt-5.4-mini").with_structured_output(CmpGrade)

def cmp_correctness(inputs, outputs, reference_outputs):
    grade = cmp_judge.invoke([
        SystemMessage(content=cmp_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric (required elements): {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}

#### Run one experiment per model. 
The target builds the agent on that model, and each row keeps the run (latency + token cost) and the correctness score.

In [ ]:
cmp_results = {}

for model_name, label in MODELS_TO_COMPARE.items():
    cmp_agent = build_shopping_agent(model=make_model(model_name))

    def run_on_model(inputs, _agent=cmp_agent):
        result = _agent.invoke(
            {"messages": [{"role": "user", "content": inputs["query"]}]},
            config={"configurable": {"thread_id": str(uuid7())}},
        )
        return {"response": result["messages"][-1].text}

    cmp_results[model_name] = client.evaluate(
        run_on_model,
        data=cmp_dataset_name,
        evaluators=[cmp_correctness],
        experiment_prefix=qualify(f"model-cmp-{model_name}"),
        metadata={"model": model_name, "size": label},
        max_concurrency=2,
    )
    print(f"{model_name} ({label}) -> {cmp_results[model_name].experiment_name}")

**Reading the result.** On these multi-step tasks you should see the models **separate** — the
larger model typically scores higher `correctness` because it completes every required element and
gets the totals and out-of-stock flags right, while the smaller model more often drops an item,
miscounts a basket total, or misses that ground beef is out of stock. The table quantifies the trade:
how much accuracy you'd give up to save on latency and tokens. If the smaller model *does* keep pace
here, that's a strong signal it's good enough for this workload. Either way you now have evidence, and
the rest of this module gives you the tools (tracing, datasets, judges, online evals) to keep making
that call as the agent evolves.

In [ ]:
import time as _time
from statistics import mean

# Summarize each experiment: efficacy (correctness) + speed (latency) come
# straight from the result rows; cost (tokens) is read back from the traces,
# because token usage is aggregated server-side and isn't on the local run
# object. Tokens land a few seconds after the run, so we retry briefly.
def _tokens_by_experiment(experiment_name, retries=6, delay=5):
    for _ in range(retries):
        runs = list(client.list_runs(project_name=experiment_name, is_root=True))
        toks = [r.total_tokens for r in runs if r.total_tokens]
        if toks:
            return mean(toks)
        _time.sleep(delay)
    return None

def summarize(results):
    scores, latencies = [], []
    for row in results:
        for r in row["evaluation_results"]["results"]:
            if r.key == "correctness" and r.score is not None:
                scores.append(r.score)
        # RunTree exposes .latency (seconds); it has no token fields.
        if row["run"].latency is not None:
            latencies.append(row["run"].latency)
    return {
        "correctness": mean(scores) if scores else None,
        "avg_latency_s": mean(latencies) if latencies else None,
        "avg_tokens": _tokens_by_experiment(results.experiment_name),
    }

print(f"{'model':<16}{'size':<16}{'correctness':>12}{'avg latency':>14}{'avg tokens':>13}")
print("-" * 71)
for model_name, label in MODELS_TO_COMPARE.items():
    s = summarize(cmp_results[model_name])
    corr = f"{s['correctness']:.0%}" if s['correctness'] is not None else "n/a"
    lat = f"{s['avg_latency_s']:.1f}s" if s['avg_latency_s'] is not None else "n/a"
    tok = f"{s['avg_tokens']:.0f}" if s['avg_tokens'] is not None else "n/a"
    print(f"{model_name:<16}{label:<16}{corr:>12}{lat:>14}{tok:>13}")

print("\nTip: open both experiments in LangSmith and use the Compare view to diff")
print("per-example correctness, latency, and cost side by side.")


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** a shopping-assistant prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = "in-store-shopping-assistant"
prompt = ChatPromptTemplate([
    ("system",
     "You are an in-store shopping assistant. "
     "Look up the item the shopper asks about, then give a concise, scannable answer: "
     "(1) the aisle, (2) whether it's in stock, (3) the price, and "
     "(4) if it's out of stock, one practical in-store substitution. "
     "Be factual; never invent an aisle or a price."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for `create_agent`, running on the workshop's shared `model`. The store's `store_directory` tool lets the agent look the item up, so the full agent loop (model → tool → model) shows up in the trace.

In [ ]:
from langchain.agents import create_agent
from agents.research_agent import store_directory

# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

agent = create_agent(model=model, tools=[store_directory], system_prompt=system_prompt)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Where do I find the tortillas, and how much are they?"}]}
)
print(result["messages"][-1].text)

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are an in-store shopping assistant. "
     "Look up the item the shopper asks about, then give a concise, scannable answer: "
     "(1) the aisle, (2) whether it's in stock, (3) the price, "
     "(4) if it's out of stock, one practical in-store substitution, "
     "and (5) flag common allergens when relevant. "
     "Be factual; never invent an aisle or a price."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt("in-store-shopping-assistant:<commit-hash>"),
# and tear down with client.delete_prompt("in-store-shopping-assistant") when you're done.

## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces.
We invoke the in-store shopping assistant (`agents/research_agent.py`) three times with **deliberately light** prompts —
each one is a single-item lookup so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.

In [ ]:
from agents.research_agent import build_shopping_agent

agent = build_shopping_agent()

warmup_prompts = [
    "What aisle is the milk in? Keep it to one lookup.",
    "Is ground beef in stock right now? Keep it to one lookup.",
    "How much are the tortillas? Keep it to one lookup.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")

## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required.
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [ ]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. 
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

### 3.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching.

In [ ]:
examples = [
    {
        "inputs": {"query": "What aisle is the milk in, and is it in stock?"},
        "outputs": {
            "reference_answer": "Milk is in aisle D2 (Dairy) and is in stock.",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "Is ground beef available? If not, what should I grab instead?"},
        "outputs": {
            "reference_answer": "Ground beef is out of stock; suggest a substitution such as ground turkey.",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "I need salsa and tomatoes — give me the aisles and prices."},
        "outputs": {
            "reference_answer": "Salsa: aisle 6, $2.79. Tomatoes: aisle 1 (Produce), $0.99.",
            "trajectory": ["task", "task"],
        },
    },
    {
        "inputs": {"query": "Plan my taco run: tortillas, ground beef, salsa. Build an aisle-ordered route and flag anything out of stock."},
        "outputs": {
            "reference_answer": "An aisle-ordered route for tortillas, ground beef (flagged out of stock), and salsa.",
            "trajectory": ["write_todos", "task", "task", "task"],
        },
    },
]

dataset_name = "modular-workshops-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")

### 3.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Run the agent and return its response plus any files it wrote.

    The evaluator receives these as separate fields so it can inspect both
    the assistant's message and the resulting file artifacts.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    return {
        "response": result["messages"][-1].text,
        "files": result.get("files") or {},
    }

In [ ]:
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant.

You will see:
1. The user's request
2. The assistant's final response
3. Any files written by the assistant
4. A success rubric

Evaluate both the response and the files. If the task asks the assistant to write a file, inspect the file contents when available. Mark `score=True` if the task was completed successfully, even if the wording differs from the rubric.

Mark `score=False` only if the task was clearly missed, the file is missing or incorrect, the response contains errors, or the assistant refused without good reason.

Give one short sentence of reasoning.
"""
judge_model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    base_url="https://gateway.smith.langchain.com/openai",
    api_key=os.environ["LANGSMITH_API_KEY_GATEWAY"],
    temperature=0
)
# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Files written: {outputs.get('files', {})}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])

    return {
        "key": "correctness",
        "score": int(grade["score"]),
        "comment": grade["reasoning"],
    }

In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [ ]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

In [ ]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an in-store shopping assistant's response satisfied the shopper's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-online-correctness",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])

## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="modular-workshops-needs-review",
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


In [ ]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "What aisle is the shredded cheese in? Keep it to one lookup.",
    "Is lettuce in stock? If not, what's a good substitute? Keep it brief.",
    "How much are the avocados and what aisle? Keep it to one lookup.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")

### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "store_directory")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper.
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.

## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the in-store shopping assistant | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.